# Week 09 Lab Work

**Dataset:** Adult / Census Income (UCI Machine Learning Repository)

1. Load the same pre-processed dataset as the last lab works.
2. Split it into training (70%), testing (15%), and validation (15%).
3. Apply SVM and KNN.
4. Compare the results with the last algorithms (the three Decision Tree variants from Week 07).

In [1]:
import os
import urllib.request

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

cols = ["age", "workclass", "fnlwgt", "education", "education-num",
        "marital-status", "occupation", "relationship", "race", "sex",
        "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]


def get_file(data_filename):
    for base in [".", ".."]:
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            if data_filename in files:
                return os.path.join(root, data_filename)

    print(f"'{data_filename}' not found in the workspace. Downloading it dynamically...")
    url = f"https://archive.ics.uci.edu/ml/machine-learning-databases/adult/{data_filename}"
    urllib.request.urlretrieve(url, data_filename)
    return data_filename


# 1. Load and pre-process the data
path = get_file("adult.data")
print(f"Loading data from: {path}")
df = pd.read_csv(path, header=None, names=cols, skipinitialspace=True, na_values="?")
df = df.dropna().drop_duplicates().reset_index(drop=True)
df["income"] = (df["income"] == ">50K").astype(int)
df = pd.get_dummies(df, drop_first=True)

print("Shape:", df.shape)
print("Missing cells:", df.isnull().sum().sum())
df.head()

Loading data from: ../Task_No_3 (week 4)/adult dataset/adult.data
Shape: (30139, 97)
Missing cells: 0


,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week,income,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,...,native-country_Portugal,native-country_Puerto-Rico,native-country_Scotland,native-country_South,native-country_Taiwan,native-country_Thailand,native-country_Trinadad&Tobago,native-country_United-States,native-country_Vietnam,native-country_Yugoslavia
0,39,77516,13,2174,0,40,0,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,50,83311,13,0,0,13,0,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,38,215646,9,0,0,40,0,False,True,False,...,False,False,False,False,False,False,False,True,False,False
3,53,234721,7,0,0,40,0,False,True,False,...,False,False,False,False,False,False,False,True,False,False
4,28,338409,13,0,0,40,0,False,True,False,...,False,False,False,False,False,False,False,False,False,False


## 2. Train, test and validation split

In [2]:
X = df.drop(columns=["income"])
y = df["income"]

# 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# split the remaining 30% into 15% test / 15% validation
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(f"Train set: {len(X_train)} rows ({len(X_train)/len(df):.1%})")
print(f"Test set:  {len(X_test)} rows ({len(X_test)/len(df):.1%})")
print(f"Val set:   {len(X_val)} rows ({len(X_val)/len(df):.1%})")

Train set: 21097 rows (70.0%)
Test set:  4521 rows (15.0%)
Val set:   4521 rows (15.0%)


## 3. SVM and KNN compared with the last algorithms

SVM and KNN use distances between rows, so the features are scaled with `StandardScaler` first. Decision Trees do not need scaling.

In [3]:
models = {
    "Decision Tree (Gini)": DecisionTreeClassifier(criterion="gini", random_state=42),
    "Decision Tree (Entropy)": DecisionTreeClassifier(criterion="entropy", random_state=42),
    "Decision Tree (Entropy, max_depth=8, pruned)": DecisionTreeClassifier(
        criterion="entropy", max_depth=8, random_state=42),
    "SVM (RBF kernel)": make_pipeline(StandardScaler(), SVC(kernel="rbf")),
    "KNN (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    val_acc = accuracy_score(y_val, model.predict(X_val))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    results.append({"Algorithm": name, "Validation Accuracy (%)": round(val_acc * 100, 2),
                    "Test Accuracy (%)": round(test_acc * 100, 2)})
    print(f"{name}: test {test_acc:.2%}")

results_df = pd.DataFrame(results).sort_values("Test Accuracy (%)", ascending=False)
results_df

Decision Tree (Gini): test 81.20%


Decision Tree (Entropy): test 81.42%
Decision Tree (Entropy, max_depth=8, pruned): test 84.56%


SVM (RBF kernel): test 83.70%


KNN (k=5): test 81.53%


,Algorithm,Validation Accuracy (%),Test Accuracy (%)
2,"Decision Tree (Entropy, max_depth=8, pruned)",85.80,84.56
3,SVM (RBF kernel),85.20,83.70
4,KNN (k=5),83.43,81.53
1,Decision Tree (Entropy),81.80,81.42
0,Decision Tree (Gini),81.71,81.20


## 4. Comparison

| Rank | Algorithm | Test Accuracy |
|---|---|---|
| 1 | Decision Tree (Entropy, max_depth=8, pruned) | 84.56% |
| 2 | SVM (RBF kernel) | 83.70% |
| 3 | KNN (k=5) | 81.53% |
| 4 | Decision Tree (Entropy) | 81.42% |
| 5 | Decision Tree (Gini) | 81.20% |

- **SVM** is the best of the two new algorithms (83.70%). It beats KNN and both unpruned Decision Trees, because the RBF kernel finds a smooth boundary that does not overfit.
- **KNN** (81.53%) is only a little better than the unpruned Decision Trees. With 96 one-hot features, distances between rows become less useful, so KNN loses accuracy.
- The **pruned Decision Tree** (max_depth=8) is still the best overall (84.56%). It is also the fastest to train and the easiest to explain.
- The validation set gives the same order for the top three, so the result is stable.